Part 1: Basic Orchestration Demo - Function calling

Part 2: Stateful Data Agent

Part 3: Interactive Data Agent

Part 4: Code Generation Agent


In [1]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


# Model

# [employees data csv](https://drive.google.com/file/d/1okdBxNrW8uokU-qLRO1VsqQk5FrRUpyv/view?usp=sharing)

> Add blockquote



In [ ]:
e

In [2]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00


In [3]:
# Import libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from typing import Dict, Any, List

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/DEPI-Cai/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
      quantization_config=bnb_config,
  torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Model loaded successfully")
print(f"✅ Device: {device}")
print(f"✅ Tokenizer loaded: {tokenizer is not None}")
print(f"✅ Model loaded: {model is not None}")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded locally from Drive
✅ Model loaded successfully
✅ Device: cuda
✅ Tokenizer loaded: True
✅ Model loaded: True


In [5]:
import json
import re
import torch

def generate_text(
    prompt,
    tokenizer,
    model,
    device,
    do_sample=False,
    temperature=0.0,
    max_new_tokens=128,
    seed=42
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature
        )

    gen_tokens = outputs[0][input_tokens:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return text

In [6]:
SYSTEM_PROMPT  = """
You are a strict decision engine.

Choose ONE action only:

CREATE_TICKET
ASK_FOR_MORE_INFO
REJECT_REQUEST

If CREATE_TICKET return:
{"action":"CREATE_TICKET","issue":"...","priority":"medium"}

Otherwise return:
{"action":"ASK_FOR_MORE_INFO"}
or
{"action":"REJECT_REQUEST"}
"""


In [7]:
def extract_json(text):
    """
    Extract first JSON safely from messy LLM output.
    """
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except:
        return None

In [8]:
def ask_llm(user_input,SYSTEM_PROMPT):
    decision_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    decision_prompt = tokenizer.apply_chat_template(
        decision_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = generate_text(
    decision_prompt,
    tokenizer,
    model,
    device,
    do_sample=False,       # 🔴 Stage 3
    temperature=0.0
    )
    return output


In [9]:
ask_llm("عندي مشكلة بالدخول للموقع",SYSTEM_PROMPT)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'{"action":"ASK_FOR_MORE_INFO"}'

# Part 2 - Stateful Data Agent

## Clean code

In [19]:
def execute_action(output, state, df):

    action = output.get("next_action")

    if action == "load_data":
        df = read_data()
        state["loaded_data"] = True
        print(df.head())

    elif action == "analysis":
        if df is None:
            print("❌ No data loaded.")
        else:
            print(df.describe())
            state["analysis"] = True

    elif action == "generate_insight":
        if df is None:
            print("❌ No data loaded.")
        else:
            summary = {
                "num_employees": len(df),
                "avg_salary": float(df["salary"].mean()),
                "max_salary": float(df["salary"].max()),
                "departments": df["department"].value_counts().to_dict()
            }

            print(summary)
            state["generated_insight"] = True

    elif action == "Finish":
        print("Workflow is finished")
        state["Finish"] = True

    else:
        print("⚠️ Unknown action:", action)

    return df, state

In [20]:
max_steps = 10
df = None

state = {
    "loaded_data": False,
    "analysis": False,
    "generated_insight": False,
    "Finish": False
}

for step in range(max_steps):

    print(f"\n=========== STEP {step+1} ===========")

    decision_prompt = build_messages(state)

    output = ask_llm(decision_prompt, SYSTEM_PROMPT)

    print(output)

    output = extract_json(output)

    df, state = execute_action(output, state, df)

    if state["Finish"]:
        break


=========== STEP 1 ===========
{
  "next_action": "load_data"
}
     name department  salary  years_experience
0   Ahmad         IT    1200                 2
1    Sara         HR    1500                 5
2  Khaled         IT    2000                 7
3    Mona    Finance    1800                 6
4    Omar         IT    2200                 8

=========== STEP 2 ===========
{
  "next_action": "analysis"
}
            salary  years_experience
count     6.000000          6.000000
mean   1683.333333          5.166667
std     381.663028          2.316607
min    1200.000000          2.000000
25%    1425.000000          3.500000
50%    1650.000000          5.500000
75%    1950.000000          6.750000
max    2200.000000          8.000000

=========== STEP 3 ===========
{
 "next_action": "generate_insight"
}
{'num_employees': 6, 'avg_salary': 1683.3333333333333, 'max_salary': 2200.0, 'departments': {'IT': 3, 'HR': 2, 'Finance': 1}}

=========== STEP 4 ===========
{
 "next_action": "Finish"


In [ ]:
# user ask ( محددة مسبقااااا) Q and get answer

In [21]:
SYSTEM_PROMPT_workflow = """ You are an AI workflow router.

Your job is to determine which workflow should be executed based on the user's question.

Available workflows:

1. sales_analysis
   Description:
   Analyze company sales data to generate insights about revenue, regions, or product performance.

2. sales_analysis_email
   Description:
   Analyze sales data and send a report email to the manager.

3. expense_analysis
   Description:
   Analyze company expenses and generate insights about spending.

4. employee_performance_analysis
   Description:
   Analyze employee performance data such as salary, experience, or department performance.

Rules:
- Read the user question carefully.
- Choose the most relevant workflow.
- Return ONLY JSON.
- Do not include explanations.
- Do not include text before or after the JSON.

JSON format:

{
  "workflow": "<workflow_name>"
}

Allowed values for workflow:
- sales_analysis
- sales_analysis_email
- expense_analysis
- employee_performance_analysis """


def full_prompt(user_input):
  decision_messages = [
        {"role": "system", "content": SYSTEM_PROMPT_workflow},
        {"role": "user", "content": user_input}
    ]

  decision_prompt = tokenizer.apply_chat_template(
      decision_messages,
      tokenize=False,
      add_generation_prompt=True
  )
  return decision_prompt



In [22]:
def employee_performance_analysis():
      max_steps = 10
      df = None

      state = {
          "loaded_data": False,
          "analysis": False,
          "generated_insight": False,
          "Finish": False
      }

      for step in range(max_steps):

          print(f"\n=========== STEP {step+1} ===========")

          decision_prompt = build_messages(state)

          output = ask_llm(decision_prompt, SYSTEM_PROMPT)

          print(output)

          output = extract_json(output)

          df, state = execute_action(output, state, df)

          if state["Finish"]:
              break

In [23]:

def fullsytem(userQ):
  decision_prompt  = full_prompt(userQ)
  output = generate_text(
        decision_prompt,
        tokenizer,
        model,
        device,
        do_sample=False,       # 🔴 Stage 3
        temperature=0.0
        )
  print(output)
  output = extract_json(output)
  if output["workflow"] == "employee_performance_analysis":
      employee_performance_analysis()
  elif output["workflow"] == "sales_analysis":
      print("sales_analysis")
  elif output["workflow"] == "sales_analysis_email":
      print("sales_analysis_email")
  elif output["workflow"] == "expense_analysis":
      print("expense_analysis")


In [24]:
fullsytem("emp workflow")

{
  "workflow": "employee_performance_analysis"
}

=========== STEP 1 ===========
{
  "next_action": "load_data"
}
     name department  salary  years_experience
0   Ahmad         IT    1200                 2
1    Sara         HR    1500                 5
2  Khaled         IT    2000                 7
3    Mona    Finance    1800                 6
4    Omar         IT    2200                 8

=========== STEP 2 ===========
{
  "next_action": "analysis"
}
            salary  years_experience
count     6.000000          6.000000
mean   1683.333333          5.166667
std     381.663028          2.316607
min    1200.000000          2.000000
25%    1425.000000          3.500000
50%    1650.000000          5.500000
75%    1950.000000          6.750000
max    2200.000000          8.000000

=========== STEP 3 ===========
{
 "next_action": "generate_insight"
}
{'num_employees': 6, 'avg_salary': 1683.3333333333333, 'max_salary': 2200.0, 'departments': {'IT': 3, 'HR': 2, 'Finance': 1}}

========

In [25]:
fullsytem("sales workflow")

{
  "workflow": "sales_analysis"
}
sales_analysis


In [26]:
fullsytem("بدي احلل المبيعات وابعت ايميل للمدير")

{
  "workflow": "sales_analysis_email"
}
sales_analysis_email


In [27]:
fullsytem("بدي احلل المبيعات ر")

{
  "workflow": "sales_analysis"
}
sales_analysis


# **[sales_dataset.csv Link](https://drive.google.com/file/d/1U5yUbg_eEAoyVdxcG-CCCRJOAhNxT8kj/view?usp=sharing)**

# Part 4  Code Generation Agent

In [58]:
# 1) userQ >> LLM >> Gen. Code  (Done)
# 2) Gen. Code  >> exec in python >> output to LLM to gen. output

import pandas as pd
df  = pd.read_csv("/content/drive/MyDrive/DEPI-Cai/TelecomTickets.csv")
df.columns

Index(['ticket_id', 'issue_type', 'priority', 'customer_plan',
       'resolution_time_minutes', 'satisfaction_score', 'ticket_date'],
      dtype='object')

In [59]:
def ask_llm(user_input,SYSTEM_PROMPT):
    decision_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    decision_prompt = tokenizer.apply_chat_template(
        decision_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = generate_text(
    decision_prompt,
    tokenizer,
    model,
    device,
    do_sample=False,       # 🔴 Stage 3
    temperature=0.0
    )
    return output


In [60]:

dataset_schema= {
    "columns": list(df.columns),
    "dftype":{  col:str(df[col].dtype) for col in df.columns}

}
dataset_schema

{'columns': ['ticket_id',
  'issue_type',
  'priority',
  'customer_plan',
  'resolution_time_minutes',
  'satisfaction_score',
  'ticket_date'],
 'dftype': {'ticket_id': 'object',
  'issue_type': 'object',
  'priority': 'object',
  'customer_plan': 'object',
  'resolution_time_minutes': 'int64',
  'satisfaction_score': 'int64',
  'ticket_date': 'object'}}

In [130]:
SYSTEM_PROMPT_CODE = ( f"""
You are a pandas code generator.

Your job is to write ONE single line of Python code that answers the user question using a pandas DataFrame named df.

dataset schema: {dataset_schema}

STRICT RULES:
- Use  ONLY existing column names of dataset_schema exactly as written
1. The DataFrame already exists and its name is: df
2. Use ONLY pandas operations on df.
3. DO NOT import anything.
4. DO NOT define functions.
5. DO NOT create variables except: result
6. The final answer MUST be stored in a variable named: result
7. Output MUST be a SINGLE line of Python code.
8. DO NOT explain anything.
9. DO NOT add comments.
10. DO NOT output markdown or formatting.

Output format example:
result = df["satisfaction_score"].mean()

Example of derived metric:
User: What is the average efficiency score?
Code: result = (df["satisfaction_score"] / df["resolution_time_minutes"]).mean()

Return ONLY the code line.
""")

In [66]:
userQ = "What is the average salary of employees?"
ask_llm(userQ,SYSTEM_PROMPT_CODE)

'result = df["salary"].mean()'

In [67]:
result = df["satisfaction_score"].mean()
result

np.float64(3.12)

In [68]:
userQ = "who have largest satisfaction_score of customers?"
ask_llm(userQ,SYSTEM_PROMPT_CODE)

"result = df.groupby('issue_type')['satisfaction_score'].max()"

In [70]:
result = df.groupby('issue_type')['satisfaction_score'].max()
result

,satisfaction_score
issue_type,
Account,5
Billing,4
Network,3
Technical,4


In [71]:
userQ = "avg satisfaction"
gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
gencode_raw

'result = df["satisfaction_score"].mean()'

In [72]:
import re

def extract_code(text):
  match = re.search(r"result\s*=.*", text)
  if match:
      return match.group(0)
  raise ValueError("No valid code found")

gencode_extract = extract_code(gencode_raw)
gencode_extract

'result = df["satisfaction_score"].mean()'

In [73]:
def execute_llm_code(code, df):

    safe_globals = {
        "__builtins__": {}
    }

    safe_locals = {
        "df": df
    }

    exec(code, safe_globals, safe_locals)

    if "result" not in safe_locals:
        raise ValueError("LLM did not return result")

    return safe_locals["result"]

In [74]:
execute_llm_code(gencode_extract, df)

np.float64(3.12)

## AST sandbox

In [127]:
allowed_function = {"mean", "count", "max", "sum", "min", "median", "std", "groupby", "agg", "nunique", "value_counts"}
forbidden_attributes = {"head", "tail", "iloc", "loc", "sample", "values", "tolist", "to_dict", "to_csv", "plot"}
import ast
def valiate_code(code):
  tree = ast.parse(code)
  for node in ast.walk(tree):
    if isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
      raise ValueError("Importing not allowed")
    if isinstance(node, (ast.While, ast.For)): # Check for both while and for loops
      raise ValueError("Loops not allowed")
    if isinstance(node, ast.FunctionDef):
      raise ValueError("Functions not allowed")

    if isinstance(node, ast.Call):
      if isinstance(node.func, ast.Attribute):
        # Check if the attribute being called is forbidden
        if node.func.attr in forbidden_attributes:
          raise ValueError(f"Forbidden operation '{node.func.attr}' not allowed")
        # Check if the attribute being called is not in the allowed list
        if node.func.attr not in allowed_function:
          raise ValueError(f"The function '{node.func.attr}' is not in allowed function list")
    elif isinstance(node, ast.Attribute): # Check for direct attribute access like df.values
        if node.attr in forbidden_attributes:
            raise ValueError(f"Forbidden attribute access '{node.attr}' not allowed")

  return True

In [78]:
valiate_code("import os")

ValueError: Importing not allowed

In [81]:
valiate_code("result = df['satisfaction_score'].mean()")

True

## full system

In [82]:
def fullsystem(df,userQ,SYSTEM_PROMPT_CODE):
  # Chech authorized??

  gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
  gencode_extract = extract_code(gencode_raw)
  print(gencode_extract)
  print("---"*20)
  # layer
  output = valiate_code(gencode_extract)
  if output:
    result = execute_llm_code(gencode_extract, df)
    return result

In [ ]:
state  Q is  authorized ture >>> q llm > gen code >> exec
state  Q is  authorized false >>> raise error

ممنوع يعرض سطر من الداتا
او يعرض اي جزء من الداتا ولو بشكل غامض
فقط يعمل فنشكن agg لعرض احصائيات عن الداتا وليس عرض عمود او row بحاله

Q >> LLM >> check authorized >>>
>>> outputLLM {"authorized":"Ture or false"} >> update state

In [99]:
state = {"authorized": False}

AUTHORIZED_SYSTEM_PROMPT =(
f"""
You are a security guard for a pandas DataFrame analysis system.

Your job is to decide whether the user's request is AUTHORIZED or UNAUTHORIZED.

The system only allows SAFE pandas aggregation queries on a DataFrame named df.
dataset schema: {dataset_schema}

AUTHORIZED requests:
- Questions that can be answered using pandas aggregation operations only.
- Examples: mean, sum, count, min, max, median, groupby aggregations.
- The request must return aggregated information, not raw rows.

UNAUTHORIZED requests include:
- Asking to import libraries
- Asking to write Python code
- Asking to use loops (for / while)
- Asking to access files or system resources
- Asking to show raw data such as df, df.head(), df.tail(), df.sample()
- Asking to display rows, records, or the entire dataset
- Asking to modify the dataframe
- Asking to create functions
- Any request unrelated to data aggregation

Decision rules:
- If the question can be answered with safe pandas aggregation only, the action is AUTHORIZED.
- If the question includes an unauthorized request, the action is UNAUTHORIZED.
- If the question is ambiguous or incomplete, the action is ASK_FOR_MORE_INFO.

Output MUST strictly follow the 'Output format' provided below.

Output format:
{{"authorized": true, "info": true}}
or
{{"authorized": false, "info": true}}
or
{{"authorized": true, "info": false}}

Do not add explanations.
Do not add extra text.
Return JSON only.
""")

In [116]:
def escalate_to_human(reason: str, user_question: str):
    """
    Simulates escalating an unauthorized request to a human supervisor.
    """
    escalation_details = {
        "tool": "escalate_to_human",
        "reason": reason,
        "user_question": user_question
    }
    print("Escalating to human supervisor:", escalation_details)
    return escalation_details

In [117]:
import re
import json

def extract_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON found in text")

    json_str = match.group(0)
    return json.loads(json_str)

In [101]:
userQ= "get avg satisfaction levels"
action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
action_raw

'{"authorized": true, "info": true}'

In [102]:
userQ= "get first five rows in data"
action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
action_raw

'{"authorized": false, "info": true}'

In [129]:
def fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT):
  # Chech authorized??
  action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
  action_raw = extract_json(action_raw)
  if action_raw["authorized"]:
      print("authorized")
      gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
      gencode_extract = extract_code(gencode_raw)
      print(gencode_extract)
      print("---"*20)
      # layer
      output = valiate_code(gencode_extract)
      if output:
        result = execute_llm_code(gencode_extract, df)
        return result
  elif action_raw["authorized"] == False:
    return escalate_to_human(reason="unauthorized_data_access", user_question=userQ)

  elif action_raw["info"] == False:
    return(fullsystem(df,input("i need more info."),SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT))

In [115]:
userQ = "avg satisfaction leve;"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)

authorized
result = df["satisfaction_score"].mean()
------------------------------------------------------------


np.float64(3.12)

In [112]:
userQ = "get first 2 rows"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)

Escalating to human supervisor: {'tool': 'escalate_to_human', 'reason': 'unauthorized_data_access', 'user_question': 'get first 2 rows'}


{'tool': 'escalate_to_human',
 'reason': 'unauthorized_data_access',
 'user_question': 'get first 2 rows'}

In [108]:
userQ = "What is the average efficiency score?"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)

authorized
result = df["satisfaction_score"].mean()
------------------------------------------------------------


np.float64(3.12)

In [119]:
userQ = "What is the average resolution time?"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)

authorized
result = df["resolution_time_minutes"].mean()
------------------------------------------------------------


np.float64(82.58)

### Test Case 1: Authorized Query (Average Resolution Time)

In [ ]:
userQ = "What is the average resolution time?"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

authorized
result = df["resolution_time_minutes"].mean()
------------------------------------------------------------
Result: 82.58


### Test Case 2: Authorized Query (Average Satisfaction Score by Issue Type)

In [ ]:
userQ = "What is the average satisfaction score by issue type?"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

authorized
result = df.groupby('issue_type')['satisfaction_score'].mean()
------------------------------------------------------------
Result: issue_type
Account      4.583333
Billing      3.538462
Network      1.923077
Technical    2.500000
Name: satisfaction_score, dtype: float64


### Test Case 3: Authorized Query (Count of tickets by priority)

In [ ]:
userQ = "What is the count of tickets by priority?"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

authorized
result = df["priority"].value_counts()
------------------------------------------------------------
Result: priority
Medium    19
High      18
Low       13
Name: count, dtype: int64


### Test Case 4: Unauthorized Query (Show raw data - should trigger escalation)

In [ ]:
userQ = "Show the first 5 tickets"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

Escalating to human supervisor: {'tool': 'escalate_to_human', 'reason': 'unauthorized_data_access', 'user_question': 'Show the first 5 tickets'}
Result: {'tool': 'escalate_to_human', 'reason': 'unauthorized_data_access', 'user_question': 'Show the first 5 tickets'}


### Test Case 5: Unclear Query (Should ask for more info)

In [ ]:
userQ = "Tell me about the tickets"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

Escalating to human supervisor: {'tool': 'escalate_to_human', 'reason': 'unauthorized_data_access', 'user_question': 'Tell me about the tickets'}
Result: {'tool': 'escalate_to_human', 'reason': 'unauthorized_data_access', 'user_question': 'Tell me about the tickets'}


### Test Case 6: Bonus Challenge (nunique - How many unique issue types exist?)

In [ ]:
userQ = "How many unique issue types exist?"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

authorized
result = df["issue_type"].nunique()
------------------------------------------------------------
Result: 4


### Test Case 7: Derived Metric (Average Efficiency Score)

In [ ]:
userQ = "What is the average efficiency score?"
result = fullsystem(df, userQ, SYSTEM_PROMPT_CODE, AUTHORIZED_SYSTEM_PROMPT)
print(f"Result: {result}")

authorized
result = (df["satisfaction_score"] / df["resolution_time_minutes"]).mean()
------------------------------------------------------------
Result: 0.07346568252414787


# Tasks: Secure AI Data Agent with Human Escalation

## Data : [[Data Link](https://drive.google.com/file/d/10vnBAtdWYFKJahOzG-70xXCLhv3bbawG/view?usp=sharing)]

## Objective

In this lab, you will extend the **Secure AI Data Agent** built in the previous session to work with a new dataset: **Telecom Support Tickets**.

Your system must:

* Allow **only aggregation-based analysis**
* Prevent exposure of **raw data**
* Validate generated code using **AST security checks**
* Use a **Tool Calling Agent** to escalate unauthorized requests
* Ask the user for clarification when the question is **unclear**

---

# Dataset Description

The dataset represents **customer support tickets** from a telecom company.

Each row represents a **support ticket**.

| Column                  | Description                                           |
| ----------------------- | ----------------------------------------------------- |
| ticket_id               | Unique identifier for the ticket                      |
| issue_type              | Type of issue (Network, Billing, Technical, Account)  |
| priority                | Ticket priority (Low, Medium, High)                   |
| customer_plan           | Customer subscription plan (Basic, Standard, Premium) |
| resolution_time_minutes | Time required to resolve the issue                    |
| satisfaction_score      | Customer satisfaction rating (1–5)                    |
| ticket_date             | Date when the ticket was created                      |

---

# System Architecture

Your system must follow this pipeline:

```
User Question
      ↓
Intent & Clarity Guard LLM
      ↓
Decision Layer
      ↓
----------------------------------
CASE 1 — AUTHORIZED
----------------------------------
Code Generator LLM
      ↓
AST Security Validation
      ↓
Safe Code Execution
      ↓
Return Result


----------------------------------
CASE 2 — UNAUTHORIZED
----------------------------------
Tool Calling Agent
      ↓
Escalate to Human Supervisor


----------------------------------
CASE 3 — ASK_FOR_MORE_INFO
----------------------------------
Return clarification question to user
```

---

# Task 1 — Update the Dataset

Replace the previous **sales dataset** with the **Telecom Support Tickets dataset**.

Ensure your system correctly recognizes these columns:

```
ticket_id
issue_type
priority
customer_plan
resolution_time_minutes
satisfaction_score
ticket_date
```

---

# Task 2 — Update the Code Generation Prompt

Update the code-generation prompt to reflect the **Telecom dataset schema**.

Generated pandas code must follow these rules:

* Use **pandas only**
* Do **not import libraries**
* Do **not create functions**
* Do **not use loops**
* Store the final result in:

```
result
```

---

# Task 3 — Security Rules

The system must **only allow aggregation operations**.

### Allowed Operations

```
mean
sum
count
min
max
median
std
groupby
agg
```

### Forbidden Operations

```
head
tail
iloc
loc
sample
values
tolist
to_dict
to_csv
```

If any forbidden operation appears in generated code, execution must stop.

---

# Task 4 — AST Code Validation

Before executing generated code, analyze it using **Python AST**.

The validator must ensure:

* No imports
* No loops
* No forbidden pandas operations
* Only safe column access

If a violation is detected, execution must be blocked.

---

# Task 5 — Derived Metric (Additional Complexity)

Extend the system to support **derived metrics**.

Example metric:

```
efficiency_score = satisfaction_score / resolution_time_minutes
```

Example question:

```
What is the average efficiency score?
```

Expected code:

```
result = (df["satisfaction_score"] / df["resolution_time_minutes"]).mean()
```

---

# Task 6 — Guard LLM Decision Layer

Before generating code, the **Guard LLM must classify the question** into one of three actions.

### Possible Outputs

```
AUTHORIZED
UNAUTHORIZED
ASK_FOR_MORE_INFO
```

---

### Case 1 — AUTHORIZED

The question clearly asks for **aggregation-based analysis**.

Example:

```
What is the average resolution time?
```

Output:

```
{"action":"AUTHORIZED"}
```

---

### Case 2 — UNAUTHORIZED

The user attempts to **retrieve raw data**.

Examples:

```
Show the first 5 tickets
Display rows from the dataset
Give me a sample of tickets
Return all satisfaction scores
```

Output:

```
{"action":"UNAUTHORIZED"}
```

---

### Case 3 — ASK_FOR_MORE_INFO

The question is **ambiguous or incomplete**.

Examples:

```
Tell me about the tickets
Analyze the dataset
What insights can you give?
```

Output:

```
{"action":"ASK_FOR_MORE_INFO"}
```

System response example:

```
Could you clarify your request?
For example, you can ask about averages, counts, or comparisons.
```

---

# Task 7 — Tool Calling Agent (Human Escalation)

If the request is **UNAUTHORIZED**, the system must escalate it to a **human supervisor**.

Create a tool named:

```
escalate_to_human
```

### Tool Input

```
{
  "reason": "unauthorized_data_access",
  "user_question": "<original question>"
}
```

---

### Example Flow

User Question:

```
Show the first 5 tickets
```

Guard LLM Output:

```
{"action":"UNAUTHORIZED"}
```

Agent Response:

```
{
  "tool": "escalate_to_human",
  "reason": "unauthorized_data_access",
  "user_question": "Show the first 5 tickets"
}
```

---

# Task 8 — System Testing

Test the system using **authorized**, **unauthorized**, and **unclear** queries.

---

## Authorized Queries

Examples:

```
What is the average resolution time?
What is the average satisfaction score by issue type?
What is the count of tickets by priority?
What is the average resolution time for Network issues?
```

---

## Unauthorized Queries

Examples:

```
Show the first 5 tickets
Display some rows from the dataset
Give me a random sample of tickets
Return all resolution times as a list
```

These must trigger the **human escalation tool**.

---

## Unclear Queries

Examples:

```
Tell me about the tickets
Analyze this dataset
What insights do you have?
```

The system must respond with a **clarification request**.

---

# Expected Example

### User Question

```
What is the average resolution time?
```

### Generated Code

```
result = df["resolution_time_minutes"].mean()
```

### Output

```
102.4
```

---

# Bonus Challenge (Optional)

Extend the allowed operations to include:

```
nunique
```

Example question:

```
How many unique issue types exist?
```

Expected code:

```
result = df["issue_type"].nunique()
```
